##  Sequence Design/Structure Prediction: 
### LigandMPNN <-> Rosetta FastRelax -> RaptorX-Single -> Chai1


### User Inputs & Global Functions

In [31]:
import os
import subprocess
"""
THIS IS THE ONLY CELL YOU SHOULD HAVE TO UPDATE FOR EACH RUN
"""
# Run specific variables
run_id = 'test' #this will be used to name chai1-related directories
input_pdb = 'test.pdb' #only input pdb filename, NOT the path to it
init_num_seqs = 2 # number of sequences to design, will be increased exponentially in each round of lmpnn/rsfr
params_file = 'sal.params' #generate this using Rosettas mol2 to params script
pipeline_run_dir = f'/wynton/home/degradolab/souchida/degrado_lab/scripts/ian_paper_scripts/{run_id}' #change this to the root run directory
lmpnn_omit_aa = 'CM' #lmpnn omit amino acid flag for lmpnn
lmpnn_omit_aa_per_resi = f"{pipeline_run_dir}/input/1_lmpnn_in/omit_aa_per_residue/omit_aa_per_residue.json" #lmpnn omit amino acid by residue
lmpnn_fixed_resi = 'A13 A50' #lmpnn fixed residues that will not be changed
chai1_ligand_name='SAL' # any 3 letter code for ligand
chai1_ligand_smiles="CC(C)(C)C1=CC(C=[N]2CC[N]3=CC4=CC(C(C)(C)C)=CC(C(C)(C)C)=C4O[Co]32(C(CCC5=CC=CC=C5)(N(S(=O)(C6=CC=CC=C6)=O)S(=O)(C7=CC=CC=C7)=O)C)O8)=C8C(C(C)(C)C)=C1" # ligand SMILES string used for Chai1 prediction

# Analysis filtering parameters
plddt_low = 60 # lower threshold for pLDDT cutoff
rmsd_high = 4.0 # upper threshold for RMSD cutoff
min_charge = -7 # lower threshold for overall net charge
max_charge = -3 # upper threshold for overall net change

#Variables that will be used throughout the notebook, once set up for user you should not need to change this 
ligandMPNN_path = '/wynton/home/degradolab/souchida/degrado_lab/LigandMPNN' #change this to the path to your lmpnn github repo
rosetta_path = '/wynton/home/degradolab/souchida/rosetta.binary.linux.release-315' # change this to the path to your rosetta build
raptor_x_single_path = '/wynton/home/degradolab/souchida/degrado_lab/RaptorX-Single' # change this to the path to your rxs github repo
scripts_path = f'{pipeline_run_dir}/scripts' # don't change this unless you change the pipeline file system
wynton_output_path = f'{pipeline_run_dir}/output/wynton' # path to SGE job output files

'''
DO NOT CHANGE THE FOLLOWING FUNCTION BELOW
queue_on_wynton: SGE job scheduler function that allow jobs to be queued but not run until a specific job is done
Arguments:
    run_script: (string) 
        The path of the script to run.
    wait_for_jobid: (None or String, default 'None')
        Job ID of a previous job.
        If specified, the current job will be queued, but wait for the specified job to be completed before running.
Returns:
    jobid: (string) 
        The job id of the queued job.
        Use this to check the status of the job using 'qstat'.
'''
def queue_on_wynton(run_script, wait_for_jobid=None):

    
    #Finally, let's queue the job:
    if wait_for_jobid is None:
        command = ['qsub', '-cwd', str(run_script)] #subprocess takes commands as a list of strings
    else:
        command = ['qsub', '-cwd', '-hold_jid', str(wait_for_jobid), str(run_script)] 
    
    #execute:
    result = subprocess.run(command, capture_output=True)

    if result.returncode != 0:
        print("Error: Job not submitted. Please check the error message below.")
        error = result.stderr.strip()
        print(error.decode())
        return None
    
    output = result.stdout.strip()
    print(output.decode()) #turn from bytes to string and print
    print("Use the command 'qstat' in the console to check the status of your job.")
    jobid = output.decode().split()[2] #get the job id
    jobid = jobid.split('.')[0] #remove the suffix denoting an array job.
    return jobid


### Sequence Design: LigandMPNN (fixed backbone) <-> Rosetta FastRelax (flexible backbone)

In [32]:
# LigandMPNN Round 1
import subprocess
import os 

# Define the paths to the input and output directories
current_round_id = '1_lmpnn_in'
output_round_id = '1_rs_in'
input_pdb_path = f'{pipeline_run_dir}/input/{current_round_id}/{input_pdb}'
output_path = f'{pipeline_run_dir}/output/{output_round_id}'
ipynb_job_script = f'{pipeline_run_dir}/output/ipynb_job_scripts/lmpnn_rd1.sh'

bash_script = f'''
#!/bin/bash 
#export TMPDIR=/scratch
#export MYTMP=`mktemp -d`
#cd $MYTMP
#$ -S /bin/bash
#$ -o {wynton_output_path}
#$ -e {wynton_output_path}
#$ -cwd
#$ -r y
#$ -j y
#$ -l gpu_mem=8G
#$ -l h_rt=00:10:00
#$ -t 1
#$ -q gpu.q 
#$ -l hostname=qb3-atgpu*

export CUDA_VISIBLE_DEVICES=$SGE_GPU

# Anything under here can be a bash script

options="$SGE_TASK_ID"

date
hostname

echo $options

# Capture the start time
start_time=$(date +%s)

#module load CBI miniconda3/23.3.1-0-py39

conda activate ligandmpnn_env

# if running on multiple inputs, change this to a multi-pdb script (see rd2_lmpnn.job for example)
cd {ligandMPNN_path} #move into the LigandMPNN directory b/c we have to
python {ligandMPNN_path}/run.py \
        --model_type "ligand_mpnn" \
        --checkpoint_path_sc "./model_params/ligandmpnn_sc_v_32_002_16.pt" \
        --seed 111 \
        --temperature 0.2 \
        --pdb_path "{input_pdb_path}" \
        --out_folder "{output_path}" \
        --batch_size {init_num_seqs} \
        --number_of_batches 1 \
        --omit_AA "{lmpnn_omit_aa}" \
        --omit_AA_per_residue "./inputs/omit_AA_per_residue.json" \
        --fixed_residues "{lmpnn_fixed_resi}" \
        --pack_side_chains 1 \
        --pack_with_ligand_context 1 \
        --repack_everything 1 \
        --number_of_packs_per_design 1 \

python {scripts_path}/lmpnn_rsfr_rxs_scripts/remove_packed_tag.py {output_path}/packed

# Capture the end time
end_time=$(date +%s)

# Calculate the duration
duration=$((end_time - start_time))

# Output the duration
echo "Script execution time: $duration seconds"

'''

with open(ipynb_job_script, 'w') as file:
        file.write(bash_script)

job_id_lmpnn_r1 = queue_on_wynton(ipynb_job_script, wait_for_jobid=None)

Your job-array 967248.1-1:1 ("lmpnn_rd1.sh") has been submitted
Use the command 'qstat' in the console to check the status of your job.


In [33]:
# Rosetta FastRelax Round 1
import subprocess
import os 

# Define the paths to the input and output directories
current_round_id = '1_rs_in'
output_round_id = '2_lmpnn_in'
output_path = f'{pipeline_run_dir}/output/{output_round_id}'
current_run_out_folder = f'{pipeline_run_dir}/output/{output_round_id}'
ipynb_job_script = f'{pipeline_run_dir}/output/ipynb_job_scripts/rsfr_rd1.sh'

bash_script = f'''
#!/bin/bash 
#export TMPDIR=/scratch
#export MYTMP=`mktemp -d`
#cd $MYTMP
#$ -S /bin/bash
#$ -o {wynton_output_path}
#$ -e {wynton_output_path}
#$ -cwd
#$ -r y
#$ -j y
#$ -l h_rt=00:05:00
#$ -t 1-{init_num_seqs}

# Anything under here can be a bash script

options="$SGE_TASK_ID"

date
hostname

# Capture the start time
start_time=$(date +%s)

#module load CBI miniconda3/23.3.1-0-py39
conda activate ligandmpnn_env

echo $options
cd {output_path} # move into folder where we want to output our structures

# Get the PDB tag from the task ID
pdbtag=$SGE_TASK_ID

input_pdb=$(python {scripts_path}/lmpnn_rsfr_rxs_scripts/find_pdb_filename.py {pipeline_run_dir}/output/{current_round_id}/packed _$pdbtag.pdb)

# Perform Rosetta relaxation on the PDB file
{rosetta_path}/main/source/bin/relax.static.linuxgccrelease \
    -database {rosetta_path}/main/database/ \
     -in:file:s $input_pdb \
     -extra_res_fa {pipeline_run_dir}/input/params/{params_file} \
     -relax:quick \
     -constraints:cst_fa_file {pipeline_run_dir}/input/constraints/constraints.cst \
        -constraints:cst_fa_weight 5.0 \
     -out:no_nstruct_label \

python {scripts_path}/lmpnn_rsfr_rxs_scripts/create_pdb_json.py {output_path}

# Capture the end time
end_time=$(date +%s)

# Calculate the duration
duration=$((end_time - start_time))

# Output the duration
echo "Script execution time: $duration seconds"

'''

with open(ipynb_job_script, 'w') as file:
        file.write(bash_script)

job_id_rsfr_r1 = queue_on_wynton(ipynb_job_script, wait_for_jobid=job_id_lmpnn_r1)

Your job-array 967249.1-2:1 ("rsfr_rd1.sh") has been submitted
Use the command 'qstat' in the console to check the status of your job.


In [34]:
# LigandMPNN Round 2
import subprocess
import os 

# Define the paths to the input and output directories
current_round_id = '2_lmpnn_in'
output_round_id = '2_rs_in'
input_multi_pdb_path = f'{pipeline_run_dir}/output/{current_round_id}/"pdb_file_paths.json"'
output_path = f'{pipeline_run_dir}/output/{output_round_id}/'
current_run_out_folder = f'{pipeline_run_dir}/output/{output_round_id}/'
ipynb_job_script = f'{pipeline_run_dir}/output/ipynb_job_scripts/lmpnn_rd2.sh'

bash_script = f'''
#!/bin/bash 
#export TMPDIR=/scratch
#export MYTMP=`mktemp -d`
#cd $MYTMP
#$ -S /bin/bash
#$ -o {wynton_output_path}
#$ -e {wynton_output_path}
#$ -cwd
#$ -r y
#$ -j y
#$ -l gpu_mem=8G
#$ -l h_rt=00:10:00
#$ -t 1
#$ -q gpu.q 

#export OMP_NUM_THREADS=1
export CUDA_VISIBLE_DEVICES=$SGE_GPU

# Anything under here can be a bash script

options="$SGE_TASK_ID"

date
hostname

echo $options

# Capture the start time
start_time=$(date +%s)

#module load CBI miniconda3/23.3.1-0-py39

conda activate ligandmpnn_env

# if running on multiple inputs, change this to a multi-pdb script (see rd2_lmpnn.job for example)
cd {ligandMPNN_path} #move into the LigandMPNN directory b/c we have to
python {ligandMPNN_path}/run.py \
        --model_type "ligand_mpnn" \
        --checkpoint_path_sc "./model_params/ligandmpnn_sc_v_32_002_16.pt" \
        --seed 111 \
        --temperature 0.2 \
        --pdb_path_multi "{input_multi_pdb_path}" \
        --out_folder "{output_path}" \
        --batch_size {init_num_seqs} \
        --number_of_batches 1 \
        --omit_AA "{lmpnn_omit_aa}" \
        --fixed_residues "{lmpnn_fixed_resi}" \
        --pack_side_chains 1 \
        --pack_with_ligand_context 1 \
        --repack_everything 1 \
        --number_of_packs_per_design 1 \

python {scripts_path}/lmpnn_rsfr_rxs_scripts/remove_packed_tag.py {output_path}/packed
python {scripts_path}/lmpnn_rsfr_rxs_scripts/pdb_tag.py {output_path}/packed

# Capture the end time
end_time=$(date +%s)

# Calculate the duration
duration=$((end_time - start_time))

# Output the duration
echo "Script execution time: $duration seconds"

'''
 
with open(ipynb_job_script, 'w') as file:
        file.write(bash_script)

job_id_lmpnn_r2 = queue_on_wynton(ipynb_job_script, wait_for_jobid=job_id_rsfr_r1)

Your job-array 967250.1-1:1 ("lmpnn_rd2.sh") has been submitted
Use the command 'qstat' in the console to check the status of your job.


In [35]:
# Rosetta FastRelax Round 2
import subprocess
import os 
# Define the paths to the input and output directories
current_round_id = '2_rs_in'
output_round_id = '3_lmpnn_in'
output_path = f'{pipeline_run_dir}/output/{output_round_id}'
current_run_out_folder = f'{pipeline_run_dir}/output/{output_round_id}'
ipynb_job_script = f'{pipeline_run_dir}/output/ipynb_job_scripts/rsfr_rd2.sh'
rd2_num_seqs = init_num_seqs * init_num_seqs

bash_script = f'''
#!/bin/bash 
#export TMPDIR=/scratch
#export MYTMP=`mktemp -d`
#cd $MYTMP
#$ -S /bin/bash
#$ -o {wynton_output_path}
#$ -e {wynton_output_path}
#$ -cwd
#$ -r y
#$ -j y
#$ -l h_rt=00:05:00
#$ -t 1-{rd2_num_seqs}

#export OMP_NUM_THREADS=1

# Anything under here can be a bash script

options="$SGE_TASK_ID"

date
hostname

echo $options


# Capture the start time
start_time=$(date +%s)

#module load CBI miniconda3/23.3.1-0-py39
conda activate ligandmpnn_env

cd {output_path} # move into folder where we want to output our structures

# Get the PDB tag from the task ID
pdbtag=$SGE_TASK_ID

input_pdb=$(python {scripts_path}/lmpnn_rsfr_rxs_scripts/find_pdb_filename.py {pipeline_run_dir}/output/{current_round_id}/packed _$pdbtag.pdb)

# Perform Rosetta relaxation on the PDB file
{rosetta_path}/main/source/bin/relax.static.linuxgccrelease \
    -database {rosetta_path}/main/database/ \
     -in:file:s $input_pdb \
     -extra_res_fa {pipeline_run_dir}/input/params/{params_file} \
     -relax:quick \
     -constraints:cst_fa_file {pipeline_run_dir}/input/constraints/constraints.cst \
        -constraints:cst_fa_weight 5.0 \
     -out:no_nstruct_label \


python {scripts_path}/lmpnn_rsfr_rxs_scripts/create_pdb_json.py {output_path}

# Capture the end time
end_time=$(date +%s)

# Calculate the duration
duration=$((end_time - start_time))

# Output the duration
echo "Script execution time: $duration seconds"
'''

with open(ipynb_job_script, 'w') as file:
        file.write(bash_script)

job_id_rsfr_r2 = queue_on_wynton(ipynb_job_script, wait_for_jobid=job_id_lmpnn_r2)


Your job-array 967251.1-4:1 ("rsfr_rd2.sh") has been submitted
Use the command 'qstat' in the console to check the status of your job.


In [36]:
# LigandMPNN Round 3
import subprocess
import os 

# Define the paths to the input and output directories
current_round_id = '3_lmpnn_in'
output_round_id = '3_rs_in'
input_multi_pdb_path = f'{pipeline_run_dir}/output/{current_round_id}/"pdb_file_paths.json"'
output_path = f'{pipeline_run_dir}/output/{output_round_id}/'
current_run_out_folder = f'{pipeline_run_dir}/output/{output_round_id}/'
ipynb_job_script = f'{pipeline_run_dir}/output/ipynb_job_scripts/lmpnn_rd3.sh'

bash_script = f'''
#!/bin/bash 
#export TMPDIR=/scratch
#export MYTMP=`mktemp -d`
#cd $MYTMP
#$ -S /bin/bash
#$ -o {wynton_output_path}
#$ -e {wynton_output_path}
#$ -cwd
#$ -r y 
#$ -j y
#$ -l gpu_mem=8G
#$ -l h_rt=00:10:00
#$ -t 1
#$ -q gpu.q 
#$ -l hostname=qb3-atgpu*

#export OMP_NUM_THREADS=1
export CUDA_VISIBLE_DEVICES=$SGE_GPU

# Anything under here can be a bash script

options="$SGE_TASK_ID"

date
hostname

echo $options

# Capture the start time
start_time=$(date +%s)

#module load CBI miniconda3/23.3.1-0-py39

conda activate ligandmpnn_env

# if running on multiple inputs, change this to a multi-pdb script (see rd2_lmpnn.job for example)
cd {ligandMPNN_path} #move into the LigandMPNN directory b/c we have to
python {ligandMPNN_path}/run.py \
        --model_type "ligand_mpnn" \
        --checkpoint_path_sc "./model_params/ligandmpnn_sc_v_32_002_16.pt" \
        --seed 111 \
        --temperature 0.2 \
        --pdb_path_multi "{input_multi_pdb_path}" \
        --out_folder "{output_path}" \
        --batch_size {init_num_seqs} \
        --number_of_batches 1 \
        --omit_AA "{lmpnn_omit_aa}" \
        --fixed_residues "{lmpnn_fixed_resi}" \
        --pack_side_chains 1 \
        --pack_with_ligand_context 1 \
        --repack_everything 1 \
        --number_of_packs_per_design 1 \

python {scripts_path}/lmpnn_rsfr_rxs_scripts/remove_packed_tag.py {output_path}/packed
python {scripts_path}/lmpnn_rsfr_rxs_scripts/pdb_tag.py {output_path}/packed

# Capture the end time
end_time=$(date +%s)

# Calculate the duration
duration=$((end_time - start_time))

# Output the duration
echo "Script execution time: $duration seconds"
'''
 
with open(ipynb_job_script, 'w') as file:
        file.write(bash_script)

job_id_lmpnn_r3 = queue_on_wynton(ipynb_job_script, wait_for_jobid=job_id_rsfr_r2)

Your job-array 967252.1-1:1 ("lmpnn_rd3.sh") has been submitted
Use the command 'qstat' in the console to check the status of your job.


In [37]:
# Rosetta FastRelax Round 3
import subprocess
import os 
# Define the paths to the input and output directories
current_round_id = '3_rs_in'
output_round_id = 'final_out'
output_path = f'{pipeline_run_dir}/output/{output_round_id}'
current_run_out_folder = f'{pipeline_run_dir}/output/{current_round_id}'
ipynb_job_script = f'{pipeline_run_dir}/output/ipynb_job_scripts/rsfr_rd3.sh'
rd3_num_seqs = init_num_seqs * init_num_seqs * init_num_seqs

bash_script = f'''
#!/bin/bash 
#export TMPDIR=/scratch
#export MYTMP=`mktemp -d`
#cd $MYTMP
#$ -S /bin/bash
#$ -o {wynton_output_path}
#$ -e {wynton_output_path}
#$ -cwd
#$ -r y
#$ -j y
#$ -l h_rt=00:05:00
#$ -t 1-{rd3_num_seqs}

#export OMP_NUM_THREADS=1
#export CUDA_VISIBLE_DEVICES=$SGE_GPU

# Anything under here can be a bash script

options="$SGE_TASK_ID"

date
hostname

echo $options

# Capture the start time
start_time=$(date +%s)

#module load CBI miniconda3/23.3.1-0-py39
conda activate ligandmpnn_env


cd {output_path} # move into folder where we want to output our structures

# Get the PDB tag from the task ID
pdbtag=$SGE_TASK_ID


input_pdb=$(python {scripts_path}/lmpnn_rsfr_rxs_scripts/find_pdb_filename.py {pipeline_run_dir}/output/{current_round_id}/packed _$pdbtag.pdb)

cp {pipeline_run_dir}/input/constraints/constraints.cst ./

# This Rosetta command is different than the previous two rounds so packstat value can be scored
{rosetta_path}/main/source/bin/rosetta_scripts.static.linuxgccrelease \
    -database {rosetta_path}/main/database/ \
    -in:file:s $input_pdb \
    -extra_res_fa {pipeline_run_dir}/input/params/{params_file} \
    -parser:protocol {scripts_path}/lmpnn_rsfr_rxs_scripts/fast_relax_packstat_csts.xml \
    -out:no_nstruct_label \
    -overwrite \

# create single fasta file for all sequences, this file will be used later in the analysis script    
python {scripts_path}/lmpnn_rsfr_rxs_scripts/mpnn_out_to_single_file_fasta.py {current_run_out_folder}/seqs {output_path}

#create single sequence fasta files for all sequence, RaptorX-single requires a seperate fasta file for each sequence
python {scripts_path}/lmpnn_rsfr_rxs_scripts/mpnn_out_to_raptor_x_input.py {current_run_out_folder}/seqs {pipeline_run_dir}/output/rxs_predictions

# Capture the end time
end_time=$(date +%s)

# Calculate the duration
duration=$((end_time - start_time))

# Output the duration
echo "Script execution time: $duration seconds"
'''

with open(ipynb_job_script, 'w') as file:
        file.write(bash_script)

job_id_rsfr_r3 = queue_on_wynton(ipynb_job_script, wait_for_jobid=job_id_lmpnn_r3)


Your job-array 967253.1-8:1 ("rsfr_rd3.sh") has been submitted
Use the command 'qstat' in the console to check the status of your job.


### Structure Prediction & Analysis: RaptorX-Single (single chain, protein only) -> Chai1 (ligand-aware, multichain option)

In [38]:
# RaptorX-Single/rxs design analysis
import subprocess
import os 
# Define the paths to the input and output directories
previous_round_id = 'final_out'
current_output_round_id = 'rxs_predictions'
current_round_output_path = f'{pipeline_run_dir}/output/{current_output_round_id}/'
previous_round_output_path = f'{pipeline_run_dir}/output/{previous_round_id}'
ipynb_job_script = f'{pipeline_run_dir}/output/ipynb_job_scripts/raptor-x-single.sh'
input_file=f'{previous_round_output_path}/score.sc'

bash_script = f'''
#!/bin/bash 
#export TMPDIR=/scratch
#export MYTMP=`mktemp -d`
#cd $MYTMP
#$ -S /bin/bash
#$ -o {wynton_output_path}
#$ -e {wynton_output_path}
#$ -cwd
#$ -r y
#$ -j y
#$ -l hostname=qb3-atgpu*
#$ -l h_rt=1:00:00
#$ -t 1
#$ -q gpu.q 


#export OMP_NUM_THREADS=1
export CUDA_VISIBLE_DEVICES=$SGE_GPU

# Anything under here can be a bash script

options="$SGE_TASK_ID"

date
hostname

echo $options

# Capture the start time
start_time=$(date +%s)

#module load CBI miniconda3/23.3.1-0-py39
module load cuda
conda activate RaptorX-Single

cd {raptor_x_single_path}

# run RaptorX-Single predictions
python pred.py {current_round_output_path}/fastas params/RaptorX-Single-ESM1b-ESM1v-ProtTrans.pt --out_dir={current_round_output_path} --device_id=0
[[ -n "$JOB_ID" ]] && qstat -j "$JOB_ID"
python {scripts_path}/lmpnn_rsfr_rxs_scripts/rename_raptor_x_single.py {current_round_output_path}
conda deactivate

conda activate lmpnn_rs_ipynbs

cd {pipeline_run_dir}/output/analysis

# extract and create Rosetta parameter score file

# Define the path to the Rosetta executable and input/output files
input_file="{previous_round_output_path}/score.sc"
rosetta_score_output_file="totalscore_pstat.txt"

# Check if input file exists
if [[ ! -f "$input_file" ]]; then
    echo "Error: Input file $input_file does not exist."
    exit 1
fi

# Sort, extract top Rosetta totalscore and packstat, and create totalscore_packstat.txt
sort -n -k2 "$input_file" | awk '{{print $2 "\\t" $19 "\\t" $NF}}' > "$rosetta_score_output_file"

# Check if totalscore.txt and packstat.txt was created successfully
if [ ! -f "$rosetta_score_output_file" ]; then
    echo "Error: total score file not created:"
    echo "  - $rosetta_score_output_file"

    exit 1
fi

cp $rosetta_score_output_file {previous_round_output_path}

# run analysis script
python {scripts_path}/lmpnn_rsfr_rxs_scripts/lmpnn_rsfr_rxs_analysis.py {previous_round_output_path} {current_round_output_path}
mv lmpnn_rsfr_rxs_analysis.csv {run_id}_lmpnn_rsfr_rxs_analysis.csv


# run filtering script
python3 {scripts_path}/lmpnn_rsfr_rxs_scripts/filter_pdbs.py  *.csv {plddt_low} {rmsd_high} {min_charge} {max_charge}
mv lmpnn_rsfr_rxs_analysis_filtered.csv {run_id}_lmpnn_rsfr_rxs_analysis_filtered.csv

# Capture the end time
end_time=$(date +%s)

# Calculate the duration
duration=$((end_time - start_time))

# Output the duration
echo "Script execution time: $duration seconds"
'''

with open(ipynb_job_script, 'w') as file:
        file.write(bash_script)

job_id_raptor_x_single = queue_on_wynton(ipynb_job_script, wait_for_jobid=job_id_rsfr_r3)


Your job-array 967254.1-1:1 ("raptor-x-single.sh") has been submitted
Use the command 'qstat' in the console to check the status of your job.


In [39]:
#Prep, submit, analyze chai1 run
import subprocess
import os 
chai1_run_input_fasta_dir=f"{pipeline_run_dir}/output/chai1_predictions/input_fastas"
chai1_run_root_output_dir=f"{pipeline_run_dir}/output/chai1_predictions/output"
ipynb_job_dir = f"{pipeline_run_dir}/output/ipynb_job_scripts/"
ipynb_job_script = f"{ipynb_job_dir}/prep_submit_analyze_chai1.sh"
chai1_scripts_path = f"{scripts_path}/chai1_scripts"
rosetta_model_dir = f"{pipeline_run_dir}/output/final_out"
run_analysis_dir = f"{pipeline_run_dir}/output/analysis"

bash_script = f'''
#!/bin/bash
#$ -S /bin/bash
#$ -o {wynton_output_path}
#$ -cwd
#$ -j y
#$ -l h_rt=00:05:00 #change as needed

 
target_designs_csv="{pipeline_run_dir}/output/analysis/{run_id}_lmpnn_rsfr_rxs_analysis_filtered.csv"
source_fasta_file_dir="{pipeline_run_dir}/output/rxs_predictions/fastas"

python3 {scripts_path}/chai1_scripts/prep_submit_analyze_chai1.py \
    $target_designs_csv $source_fasta_file_dir \
    {chai1_run_input_fasta_dir} {chai1_run_root_output_dir} \
    "{chai1_ligand_name}" "{chai1_ligand_smiles}" \
    {ipynb_job_dir} {chai1_scripts_path} \
    {wynton_output_path} {rosetta_model_dir} {run_analysis_dir}\
 
'''

with open(ipynb_job_script, 'w') as file:
        file.write(bash_script)

job_id_run_chai1 = queue_on_wynton(ipynb_job_script, wait_for_jobid=job_id_raptor_x_single)

Your job 967255 ("prep_submit_analyze_chai1.sh") has been submitted
Use the command 'qstat' in the console to check the status of your job.


Pipeline scripts written by Ian Bakanas, analysis script/adapted to jupyter notebook by Stephanie Ouchida